# HyperKT Stage 9G — Checkpoint Verification and Conditional Protocol Freeze

**No retraining. No test-set evaluation.** Upload Stage 9E checkpoint ZIP, Stage 2 ZIP, and Stage 3B ZIP. Validates 15 checkpoints, SHA-256, strict loading, validation metrics, causal non-interference, and produces a machine-readable decision. Independent DKT implementation remains a baseline-fidelity limitation.

In [ ]:
%pip -q install numpy pandas scipy scikit-learn torch

In [ ]:
import io, os, json, zipfile, hashlib, math, random, tempfile, shutil, pathlib, collections
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, f1_score
from google.colab import files
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',DEVICE)
print('Upload THREE files: Stage 9E checkpoint folder ZIP, Stage 2 ZIP, Stage 3B ZIP')
uploaded=files.upload()
assert len(uploaded)==3, 'Upload exactly three ZIP archives.'
zs={p:zipfile.ZipFile(io.BytesIO(b)) for p,b in uploaded.items()}
for name,z in zs.items(): print(name,len(z.namelist()),'members')
z9=next(z for z in zs.values() if any(x.endswith('/protocol.json') and 'Stage9E' in x for x in z.namelist()))
z2=next(z for z in zs.values() if 'train_sequences.npz' in z.namelist())
z3=next(z for z in zs.values() if 'hypergraph_repaired.npz' in z.namelist())
root=next(x[:-len('protocol.json')] for x in z9.namelist() if x.endswith('/protocol.json'))
protocol=json.loads(z9.read(root+'protocol.json'))
sha2=next(hashlib.sha256(b).hexdigest() for p,b in uploaded.items() if zs[p] is z2)
sha3=next(hashlib.sha256(b).hexdigest() for p,b in uploaded.items() if zs[p] is z3)
assert sha2==protocol['stage2_sha256'], 'Stage 2 ZIP SHA-256 mismatch'
assert sha3==protocol['stage3b_sha256'], 'Stage 3B ZIP SHA-256 mismatch'
print('Source hashes verified')

In [ ]:
def read_npz(z,name):
    with np.load(io.BytesIO(z.read(name)),allow_pickle=False) as a: return {k:a[k].copy() for k in a.files}
train=read_npz(z2,'train_sequences.npz'); val=read_npz(z2,'val_sequences.npz')
struct=read_npz(z3,'hypergraph_repaired.npz')
manifest=json.loads(z2.read('stage2_report.json'))
N_SKILLS=int(max(train['target_skill'].max(),val['target_skill'].max(),struct['skill_ids'].max())+1)
H=np.zeros((N_SKILLS,struct['assignments'].shape[1]),dtype=np.float32)
H[struct['skill_ids'].astype(int)]=struct['assignments'].astype(np.float32)
MAX_LEN=train['history_skill'].shape[1]
print('N_SKILLS:',N_SKILLS,'MAX_LEN:',MAX_LEN,'hyperedges:',H.shape[1])
assert N_SKILLS==151 and MAX_LEN==200, 'Unexpected source configuration: review before running.'


In [ ]:
class SeqDataset(Dataset):
    def __init__(self,d): self.d=d
    def __len__(self):return len(self.d['mask'])
    def __getitem__(self,i):
        return tuple(torch.as_tensor(self.d[k][i],dtype=torch.long if k in ('history_skill','history_correct','target_skill') else torch.float32) for k in ('history_skill','history_correct','target_skill','target_correct','mask'))
TRAIN_DS=SeqDataset(train); VAL_DS=SeqDataset(val)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
    try:torch.use_deterministic_algorithms(True,warn_only=True)
    except Exception:pass
class CausalBlock(nn.Module):
    def __init__(self,d=128,heads=8,drop=.1,decay=False):
        super().__init__();self.heads=heads;self.dhead=d//heads;self.decay=decay
        self.qkv=nn.Linear(d,3*d);self.proj=nn.Linear(d,d);self.drop=nn.Dropout(drop)
        self.ln1=nn.LayerNorm(d);self.ln2=nn.LayerNorm(d)
        self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Dropout(drop),nn.Linear(4*d,d))
        if decay:self.log_rate=nn.Parameter(torch.full((heads,),-3.0))
    def forward(self,x):
        B,T,D=x.shape; z=self.ln1(x)
        q,k,v=self.qkv(z).chunk(3,dim=-1)
        def split(t):return t.reshape(B,T,self.heads,self.dhead).transpose(1,2)
        q,k,v=map(split,(q,k,v))
        logits=torch.matmul(q,k.transpose(-2,-1))*(self.dhead**-.5)
        pos=torch.arange(T,device=x.device)
        dist=(pos[:,None]-pos[None,:]).clamp(min=0).float()
        if self.decay:logits=logits-torch.nn.functional.softplus(self.log_rate)[None,:,None,None]*dist[None,None,:,:]
        future=pos[None,:]>pos[:,None]
        logits=logits.masked_fill(future[None,None,:,:],torch.finfo(logits.dtype).min)
        a=torch.softmax(logits,dim=-1)
        x=x+self.drop(self.proj(torch.matmul(self.drop(a),v).transpose(1,2).reshape(B,T,D)))
        x=x+self.drop(self.ff(self.ln2(x)))
        return x
class HyperKT(nn.Module):
    def __init__(self,n_skills,H,hyper=False,decay=False,d=128,layers=2,heads=8,drop=.1):
        super().__init__();self.hyper=hyper
        self.skill=nn.Embedding(n_skills,d,padding_idx=1)
        self.response=nn.Embedding(2,d)
        self.pos=nn.Embedding(MAX_LEN,d)
        self.blocks=nn.ModuleList([CausalBlock(d,heads,drop,decay) for _ in range(layers)])
        self.norm=nn.LayerNorm(d)
        if hyper:
            self.register_buffer('assignments',torch.tensor(H))
            self.edge_embedding=nn.Parameter(torch.randn(H.shape[1],d)*.02)
            self.fusion=nn.Linear(d*2,d)
        self.output=nn.Sequential(nn.LayerNorm(d*2),nn.Linear(d*2,d),nn.GELU(),nn.Dropout(drop),nn.Linear(d,1))
    def forward(self,hs,hc,ts):
        B,T=hs.shape; pos=torch.arange(T,device=hs.device)[None,:]
        x=self.skill(hs)+self.response(hc.clamp(0,1))+self.pos(pos)
        for layer in self.blocks:x=layer(x)
        x=self.norm(x)
        target=self.skill(ts)
        if self.hyper:
            structural=self.assignments[ts]@self.edge_embedding
            target=target+torch.tanh(self.fusion(torch.cat([target,structural],dim=-1)))
        return self.output(torch.cat([x,target],dim=-1)).squeeze(-1)


In [ ]:
class DKT(nn.Module):
    # Independent Stage 9C baseline implementation, not a certified official DKT reproduction.
    def __init__(self,n,d=128,dropout=.1):
        super().__init__();self.n=n;self.emb=nn.Embedding(n*2,d)
        self.rnn=nn.LSTM(d,d,batch_first=True);self.drop=nn.Dropout(dropout);self.out=nn.Linear(d,n)
    def forward(self,hs,hc,ts):
        h,_=self.rnn(self.emb(hs+hc.clamp(0,1)*self.n))
        return self.out(self.drop(h)).gather(-1,ts.unsqueeze(-1)).squeeze(-1)

class HyperKTGated(HyperKT):
    # Only change vs original HyperKT: a scalar learned gate on the hypergraph residual.
    def __init__(self,n_skills,H,d=128,layers=2,heads=8,drop=.1):
        super().__init__(n_skills,H,hyper=True,decay=True,d=d,layers=layers,heads=heads,drop=drop)
        self.hyper_gate=nn.Parameter(torch.tensor(-2.0))
    def forward(self,hs,hc,ts):
        B,T=hs.shape;pos=torch.arange(T,device=hs.device)[None,:]
        x=self.skill(hs)+self.response(hc.clamp(0,1))+self.pos(pos)
        for layer in self.blocks:x=layer(x)
        x=self.norm(x);target=self.skill(ts)
        structural=self.assignments[ts]@self.edge_embedding
        target=target+torch.sigmoid(self.hyper_gate)*torch.tanh(self.fusion(torch.cat([target,structural],dim=-1)))
        return self.output(torch.cat([x,target],dim=-1)).squeeze(-1)

def make_model(name):
    if name=='DKT_matched':return DKT(N_SKILLS).to(DEVICE)
    if name=='HyperKT_matched':return HyperKT(N_SKILLS,H,hyper=True,decay=True).to(DEVICE)
    if name=='HyperKT_gated':return HyperKTGated(N_SKILLS,H).to(DEVICE)
    raise ValueError(name)



In [ ]:
CFG=protocol['config']; SEEDS=protocol['seeds']; VARIANTS=protocol['models']
assert SEEDS==[42,123,2024,3407,7777]
assert set(VARIANTS)=={'DKT_matched','HyperKT_matched','HyperKT_gated'}
assert len(val['mask'])==445 and int(val['mask'].sum())==23858
print('Protocol:',json.dumps(protocol,indent=2))

In [ ]:
def metrics(y,p):
    y=np.asarray(y,dtype=np.int64);p=np.clip(np.asarray(p,dtype=np.float64),1e-7,1-1e-7)
    edges=np.linspace(0,1,16);inds=np.minimum(np.digitize(p,edges[1:-1]),14)
    ece=sum((inds==i).mean()*abs(y[inds==i].mean()-p[inds==i].mean()) for i in range(15) if np.any(inds==i))
    return dict(auc=float(roc_auc_score(y,p)),nll=float(log_loss(y,p,labels=[0,1])),brier=float(brier_score_loss(y,p)),f1=float(f1_score(y,p>=.5,zero_division=0)),ece15=float(ece),n=int(len(y)))
def loader(ds,shuffle,seed):
    g=torch.Generator().manual_seed(seed)
    return DataLoader(ds,batch_size=CFG['batch_size'],shuffle=shuffle,generator=g,num_workers=0,pin_memory=True)
def evaluate(model,dl):
    model.eval();Y=[];P=[]
    with torch.no_grad():
        for hs,hc,ts,y,mask in dl:
            hs,hc,ts=hs.to(DEVICE),hc.to(DEVICE),ts.to(DEVICE)
            p=torch.sigmoid(model(hs,hc,ts)).cpu().numpy(); m=mask.numpy().astype(bool)
            Y.append(y.numpy()[m]);P.append(p[m])
    return metrics(np.concatenate(Y),np.concatenate(P))


In [ ]:
import hashlib
results=[]; problems=[]; probe=[]
val_loader=loader(VAL_DS,False,42)
for name in VARIANTS:
 for seed in SEEDS:
  prefix=f'{root}{name}_seed{seed}/'
  key=prefix+'best.pt'; metric_key=prefix+'best_metrics.json'
  try:
   assert key in z9.namelist(),f'Missing {key}'
   assert metric_key in z9.namelist(),f'Missing {metric_key}'
   raw=z9.read(key); digest=hashlib.sha256(raw).hexdigest()
   ckpt=torch.load(io.BytesIO(raw),map_location='cpu',weights_only=True)
   assert ckpt['variant']==name and ckpt['seed']==seed
   assert ckpt['config']==CFG
   assert ckpt['stage2_sha256']==sha2 and ckpt['stage3b_sha256']==sha3
   assert all(torch.isfinite(v).all().item() for v in ckpt['state_dict'].values() if torch.is_tensor(v))
   model=make_model(name);model.load_state_dict(ckpt['state_dict'],strict=True);model.eval()
   saved=json.loads(z9.read(metric_key))
   assert saved['best_epoch']==ckpt['best_epoch']
   actual=evaluate(model,val_loader)
   diffs={k:abs(float(actual[k])-float(saved[k])) for k in ('auc','nll','brier','f1','ece15')}
   assert actual['n']==saved['n']==23858
   maxdiff=max(diffs.values())
   assert maxdiff<1e-5, f'Validation mismatch {maxdiff}'
   hs=torch.as_tensor(val['history_skill'][:2],device=DEVICE,dtype=torch.long)
   hc=torch.as_tensor(val['history_correct'][:2],device=DEVICE,dtype=torch.long)
   ts=torch.as_tensor(val['target_skill'][:2],device=DEVICE,dtype=torch.long)
   with torch.no_grad():
    base=model(hs,hc,ts)
    changed_hc=hc.clone();changed_ts=ts.clone()
    changed_hc[:,100:]=1-changed_hc[:,100:]
    changed_ts[:,100:]=(changed_ts[:,100:]+2)%N_SKILLS
    altered=model(hs,changed_hc,changed_ts)
    delta=(base[:,:100]-altered[:,:100]).abs().max().item()
   assert delta<1e-5,f'Causal non-interference failure: {delta}'
   results.append({'variant':name,'seed':seed,'sha256':digest,'best_epoch':ckpt['best_epoch'],'max_metric_difference':maxdiff,'causal_delta':delta,**actual})
   print('PASS',name,seed,'AUC',round(actual['auc'],6),'maxdiff',round(maxdiff,9))
   del model
  except Exception as exc:
   problems.append({'variant':name,'seed':seed,'error':repr(exc)})
   print('FAIL',name,seed,repr(exc))
assert len(results)+len(problems)==15
print('Passed:',len(results),'Failed:',len(problems))

In [ ]:
out=Path('/content/hyperkt_stage9g');out.mkdir(exist_ok=True)
pd.DataFrame(results).to_csv(out/'checkpoint_verification.csv',index=False)
(out/'errors.json').write_text(json.dumps(problems,indent=2))
summary=pd.DataFrame(results).groupby('variant')[['auc','nll','brier','f1','ece15']].agg(['mean','std']) if results else pd.DataFrame()
summary.to_csv(out/'recomputed_summary.csv')
lock={'stage':'9G','source_protocol':protocol,'checkpoints_verified':len(results),'checkpoint_failures':problems,'validation_tested_only':True,'test_set_used':False,'selected_proposed_model':'HyperKT_matched','primary_comparator':'DKT_matched','negative_ablation':'HyperKT_gated','source_hashes_verified':True,'baseline_fidelity':'independent DKT implementation; not certified official reproduction','hypergraph_seeds':'fixed pretrained Stage3B hypergraph','event_uniqueness':'Stage9F previously reported pass; not independently rerun here','decision': 'CONDITIONAL_LOCK_PENDING_BASELINE_FIDELITY_REVIEW' if len(results)==15 and not problems else 'BLOCKED_CHECKPOINT_FAILURE'}
(out/'protocol_lock.json').write_text(json.dumps(lock,indent=2))
print(json.dumps(lock,indent=2))
import zipfile
with zipfile.ZipFile('/content/hyperkt_stage9g.zip','w',zipfile.ZIP_DEFLATED) as z:
 for f in out.iterdir():z.write(f,f.name)
files.download('/content/hyperkt_stage9g.zip')